# Multi-agent DDPG for MPE (Multi-Agent Particle Environment)


* MADDPG: Multi-Agent Deep Deterministic Policy Gradient (Lowe et al., 2017) [paper](https://arxiv.org/abs/1706.02275)
* MPE: Multi-Agent Particle Environment using pettingzoo library [github](https://github.com/Farama-Foundation/PettingZoo)
* DDPG: Deep Deterministic Policy Gradient (Lillicrap et al., 2015) [paper](https://arxiv.org/abs/1509.02971)

## MADDPG Algorithm

**Multi-agent DDPG (MADDPG)** (Lowe et al., 2017) extends DDPG to an environment where multiple agents are coordinating to complete tasks with only local information. In the viewpoint of one agent, the environment is non-stationary as policies of other agents are quickly upgraded and remain unknown. MADDPG is an actor-critic model redesigned particularly for handling such a changing environment and interactions between agents.

The problem can be formalized in the multi-agent version of MDP, also known as _Markov games_. MADDPG is proposed for partially observable Markov games. Say, there are $N$ agents in total with a set of states $\mathcal{S}$. Each agent owns a set of possible actions, $\mathcal{A}_1, \dots, \mathcal{A}_N$, and a set of observation, $\mathcal{O}_1, \dots, \mathcal{O}_N$. The state transition function involves all states, action and observation spaces $\mathcal{T}: \mathcal{S} \times \mathcal{A}_1 \times ... \times \mathcal{A}_N \rightarrow \mathcal{S}$. Each agent's stochastic policy only involves its own state and action: $\pi_{\theta_i}: \mathcal{O}_i \times \mathcal{A}_i \mapsto [0, 1]$, a probability distribution over actions given its own observation, or a deterministic policy: $\mu_{\theta_i}: \mathcal{O}_i \mapsto \mathcal{A}_i$.

Let $\vec{o} = {o_1, \dots, o_N}, \vec{\mu} = {\mu_1, \dots, \mu_N}$ and the policies are parameterized by $\vec{\theta} = {\theta_1, \dots, \theta_N}$.

The critic in MADDPG learns a centralized action-value function $Q^\mu_i(\vec{o}, a_1, \dots, a_N)$ for the i-th agent, where $a_1 \in \mathcal{A}_1, \dots, a_N \in \mathcal{A}_N$ are actions of all agents. Each $Q^\mu_i$ is learned separately for $i=1, \dots, N$ and therefore multiple agents can have arbitrary reward structures, including conflicting rewards in a competitive setting. Meanwhile, multiple actors, one for each agent, are exploring and upgrading the policy parameters $\theta_i$ on their own.

### Actor update:

$$
\nabla_{\theta_i} J(\theta_i) = \mathbb{E}_{\vec{o}, a \sim \mathcal{D}} [\nabla_{a_i} Q^{\vec{\mu}}_i (\vec{o}, a_1, \dots, a_N) \nabla_{\theta_i} \mu_{\theta_i}(o_i) \rvert_{a_i=\mu_{\theta_i}(o_i)} ]
$$

Where $\mathcal{D}$ is the memory buffer for experience replay, containing multiple episode samples $(\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}’)$ — given current observation $\vec{o}$, agents take action $a_1, \dots, a_N$ and get rewards $rr_1, \dots, r_N$, leading to the new observation $\vec{o}’$.

### Critic update:

$$
\begin{aligned}
\mathcal{L}(\theta_i) &= \mathbb{E}_{\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}'}[ (Q^{\vec{\mu}}_i(\vec{o}, a_1, \dots, a_N) - y)^2 ] & \\
\text{where } y &= r_i + \gamma Q^{\vec{\mu}'}_i (\vec{o}', a'_1, \dots, a'_N) \rvert_{a'_j = \mu'_{\theta_j}} & \scriptstyle{\text{; TD target!}}
\end{aligned}
$$
 
where $\vec{\mu}’$ are the target policies with delayed softly-updated parameters.

If the policies $\vec{\mu}$ are unknown during the critic update, we can ask each agent to learn and evolve its own approximation of others' policies. Using the approximated policies, MADDPG still can learn efficiently although the inferred policies might not be accurate.

To mitigate the high variance triggered by the interaction between competing or collaborating agents in the environment, MADDPG proposed one more element - _policy ensembles_:

1. Train K policies for one agent;
2. Pick a random policy for episode rollouts;
3. Take an ensemble of these K policies to do gradient update.

In summary, MADDPG added three additional ingredients on top of DDPG to make it adapt to the multi-agent environment:

* Centralized critic + decentralized actors;
* Actors are able to use estimated policies of other agents for learning;
* Policy ensembling is good for reducing variance.

<div style="text-align:center"><img src="../../assets/images/MADDPG.png" width="600" height="auto"></div>


Here is the final algorithm:

<div style="text-align:center"><img src="../../assets/images/MADDPG-algorithm.png" width="600" height="auto"></div>

### ENVIRONMENT

We going to use PettingZoo library to create the environment. And our environment is MPE (Multi-Agent Particle Environment). The environment is a simple grid-world with agents and landmarks.



In [ ]:
TODO: 
1. Model 
2. ReplayBuffer
3. Agent 
4. MultiAgent 